# MAPPO Distance Autocurriculum 250x250 Healthy Reset

Run the known-healthy 250x250 distance-autocurriculum rescue recipe from the sudden-drop investigation. The checked-in default reproduces the 10-update sanity run that kept the source boundary stable and trended upward.


In [ ]:
from pathlib import Path
import os
import sys

# Set these before importing JAX in this kernel.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.35")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "ant_byte_env").exists():
    raise RuntimeError("Launch this notebook from the cool-antz repo or a subdirectory.")
os.chdir(PROJECT_ROOT)

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from ant_byte_env import notebook_workflows as workflows

runtime_status = workflows.configure_jax_notebook_runtime()
workflows.assert_notebook_resources_available(runtime_status)
runtime_status


In [ ]:
import importlib

import jax

from ant_byte_env import notebook_workflows as workflows
from ant_byte_env.training.jax_mappo import runner as jax_runner

workflows = importlib.reload(workflows)
jax_runner = importlib.reload(jax_runner)
print(f"JAX device: {jax.devices()[0]}")


## Quick Smoke Run

Run one tiny job before starting the 250x250 rescue run.


In [ ]:
smoke_metrics = workflows.run_jax_smoke(jax_runner.main)
smoke_metrics


## Healthy Distance-Rescue Settings

Edit `experiments/half_scale_distance_autocurriculum_250x250_healthy_reset.json` for durable changes. The important safety knobs are `critic_architecture=set_cnn`, `distance_autocurriculum_success_cookies=0`, `distance_progress_normalizer=stage`, `reset_env_each_update=True`, `reset_optimizer_on_load=True`, and `learning_rate=2.5e-5`.


In [ ]:
DISTANCE_CONFIG = PROJECT_ROOT / "experiments" / "half_scale_distance_autocurriculum_250x250_healthy_reset.json"
experiment = workflows.load_jax_experiment(DISTANCE_CONFIG)
experiment_args = dict(experiment.args)

RUN_DIR = workflows.resolve_project_path(PROJECT_ROOT, experiment_args["run_dir"])
MEDIA_DIR = RUN_DIR / "media"
SOURCE_CHECKPOINT = workflows.resolve_project_path(PROJECT_ROOT, experiment_args["load_model"])
if not SOURCE_CHECKPOINT.exists():
    raise FileNotFoundError(f"Run or restore the source checkpoint first: {SOURCE_CHECKPOINT}")
experiment_args["load_model"] = str(SOURCE_CHECKPOINT)

UPDATE_TIMESTEPS = workflows.update_timesteps(
    num_envs=int(experiment_args["num_envs"]),
    num_steps=int(experiment_args["num_steps"]),
)
GLOBAL_UPDATE_CAP = int(experiment.metadata.get("default_updates", 10))
CHECKPOINT_NAME = Path(str(experiment_args.get("save_model", "model.pkl"))).name
ROLLOUT_TILE_SIZE = workflows.NOTEBOOK_ROLLOUT_TILE_SIZE
ROLLOUT_POLICY_TEMPERATURE = workflows.notebook_rollout_policy_temperature(experiment.metadata)
CHECKPOINT_VIDEO_INTERVAL_UPDATES = experiment.metadata.get("checkpoint_video_interval_updates")
CHECKPOINT_VIDEO_MAX_FRAMES = experiment.metadata.get("checkpoint_video_max_frames")
CHECKPOINT_VIDEO_POLICY_TEMPERATURE = workflows.notebook_rollout_policy_temperature(
    experiment.metadata,
    key="checkpoint_video_policy_temperature",
)
CHECKPOINT_VIDEO_RENDER_STYLE = experiment.metadata.get("checkpoint_video_render_style")
CHECKPOINT_VIDEO_SHOW_VISION = bool(experiment.metadata.get("checkpoint_video_show_vision", True))
CHECKPOINT_VIDEO_WANDB_KEY_PREFIX = str(experiment.metadata["checkpoint_video_wandb_key_prefix"])
ROLLOUT_RENDER_STYLE = experiment.metadata.get("rollout_render_style")
ROLLOUT_SHOW_VISION = bool(experiment.metadata.get("rollout_show_vision", True))

WANDB_PROJECT = "cool-antz"
WANDB_ENTITY = None
WANDB_GROUP = experiment.name
WANDB_RUN_NAME = experiment.name
WANDB_MODE = "online"

COMMON_ARGS = workflows.config_common_args(
    experiment_args,
    exclude=workflows.SINGLE_CHECKPOINT_ARG_EXCLUDES,
)
COMMON_ARGS += [
    "--wandb-mode",
    WANDB_MODE,
    "--wandb-group",
    WANDB_GROUP,
    "--wandb-run-name",
    WANDB_RUN_NAME,
    "--wandb-tags",
    "distance-autocurriculum",
    "250x250",
    "set-cnn",
    "reset-env-each-update",
    "reset-optimizer-on-load",
    "stage-normalized-distance",
    "healthy-rescue",
]
if WANDB_PROJECT is not None:
    COMMON_ARGS += ["--wandb-project", WANDB_PROJECT]
if WANDB_ENTITY is not None:
    COMMON_ARGS += ["--wandb-entity", WANDB_ENTITY]
if experiment.metadata.get("notes"):
    COMMON_ARGS += ["--wandb-notes", str(experiment.metadata["notes"])]

{
    "experiment": experiment.name,
    "config": DISTANCE_CONFIG,
    "source_checkpoint": SOURCE_CHECKPOINT,
    "run_dir": RUN_DIR,
    "updates": GLOBAL_UPDATE_CAP,
    "update_timesteps": UPDATE_TIMESTEPS,
    "total_timesteps": GLOBAL_UPDATE_CAP * UPDATE_TIMESTEPS,
    "num_ants": experiment_args.get("num_ants"),
    "arena": (experiment_args.get("width"), experiment_args.get("height")),
    "critic_architecture": experiment_args.get("critic_architecture"),
    "learning_rate": experiment_args.get("learning_rate"),
    "distance_autocurriculum_success_cookies": experiment_args.get("distance_autocurriculum_success_cookies"),
    "distance_progress_normalizer": experiment_args.get("distance_progress_normalizer"),
    "reset_env_each_update": experiment_args.get("reset_env_each_update"),
    "reset_optimizer_on_load": experiment_args.get("reset_optimizer_on_load"),
    "healthy_control_first_update_deliveries": experiment.metadata.get("healthy_control_first_update_deliveries"),
    "healthy_control_final_update_deliveries": experiment.metadata.get("healthy_control_final_update_deliveries"),
}


## Train Healthy Rescue


In [ ]:
training_result = workflows.run_jax_checkpoint_training(
    run_dir=RUN_DIR,
    common_args=COMMON_ARGS,
    update_timesteps=UPDATE_TIMESTEPS,
    global_update_cap=GLOBAL_UPDATE_CAP,
    train_main=jax_runner.main,
    checkpoint_name=CHECKPOINT_NAME,
    progress_label="distance rescue",
    checkpoint_video_interval_updates=CHECKPOINT_VIDEO_INTERVAL_UPDATES,
    checkpoint_video_max_frames=CHECKPOINT_VIDEO_MAX_FRAMES,
    checkpoint_video_policy_temperature=CHECKPOINT_VIDEO_POLICY_TEMPERATURE,
    checkpoint_video_wandb_key_prefix=CHECKPOINT_VIDEO_WANDB_KEY_PREFIX,
    checkpoint_video_render_style=CHECKPOINT_VIDEO_RENDER_STYLE,
    checkpoint_video_show_vision=CHECKPOINT_VIDEO_SHOW_VISION,
)
FINAL_CHECKPOINT_PATH = training_result["checkpoint_path"]
ROLLOUT_CHECKPOINT_PATH = FINAL_CHECKPOINT_PATH
training_result


## Optional Local Render and Vault


In [ ]:
# rollout_result = workflows.render_jax_checkpoint_rollout(
#     run_dir=RUN_DIR,
#     checkpoint_path=ROLLOUT_CHECKPOINT_PATH,
#     media_dir=MEDIA_DIR,
#     rollout_filename="jax_mappo_hs250_distance_auto_healthy_reset_rollout.mp4",
#     title="JAX MAPPO 250x250 distance-autocurriculum healthy-reset rollout",
#     description="Sampled rollout from the healthy distance-autocurriculum reset recipe.",
#     metadata={
#         "experiment_config": str(DISTANCE_CONFIG),
#         "source_checkpoint": str(SOURCE_CHECKPOINT),
#         "critic_architecture": experiment_args.get("critic_architecture"),
#         "learning_rate": experiment_args.get("learning_rate"),
#         "distance_autocurriculum_success_cookies": experiment_args.get("distance_autocurriculum_success_cookies"),
#         "distance_progress_normalizer": experiment_args.get("distance_progress_normalizer"),
#         "reset_env_each_update": experiment_args.get("reset_env_each_update"),
#         "reset_optimizer_on_load": experiment_args.get("reset_optimizer_on_load"),
#     },
#     max_frames=1200,
#     tile_size=ROLLOUT_TILE_SIZE,
#     policy_temperature=ROLLOUT_POLICY_TEMPERATURE,
#     render_style=ROLLOUT_RENDER_STYLE,
#     show_vision=ROLLOUT_SHOW_VISION,
#     wandb_project=WANDB_PROJECT,
#     wandb_entity=WANDB_ENTITY,
#     wandb_group=WANDB_GROUP,
#     wandb_run_name=f"{WANDB_GROUP}_rollout",
#     wandb_mode="disabled",
#     wandb_video_key=None,
#     wandb_step=training_result["final_train_metrics"].get("global_step"),
# )
# rollout_result
